In [1]:
from pytorch_forecasting import TimeSeriesDataSet
import pandas as pd

def rebuild_reference_dataset(df: pd.DataFrame, params: dict) -> TimeSeriesDataSet:
    """
    Rebuild the reference TimeSeriesDataSet using the same structure as during training.

    Parameters:
    - df: full input DataFrame with 'Datetime', 'Consommation', and all features
    - params: dict of training parameters (same as used in training phase)

    Returns:
    - TimeSeriesDataSet object for inference use
    """
    df = df.copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values("Datetime").reset_index(drop=True)
    df["time_idx"] = range(len(df))
    df["series_id"] = "France"

    dataset = TimeSeriesDataSet(
        df,
        time_idx="time_idx",
        target="Consommation",
        group_ids=["series_id"],
        max_encoder_length=params.get("max_encoder_length", 288),
        max_prediction_length=params.get("max_prediction_length", 48),
        time_varying_known_reals=params.get("known_reals", []),
        time_varying_unknown_reals=params.get("unknown_reals", []),
        time_varying_known_categoricals=params.get("known_categoricals", []),
        time_varying_unknown_categoricals=params.get("unknown_categoricals", []),
        static_reals=params.get("static_reals", []),
        static_categoricals=params.get("static_categoricals", []),
        target_normalizer=params.get("target_normalizer", None),
        add_relative_time_idx=params.get("add_relative_time_idx", True),
        add_target_scales=params.get("add_target_scales", True),
        add_encoder_length=params.get("add_encoder_length", True)
    )

    return dataset

In [2]:
import torch
import gc
import pandas as pd
import logging
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import MAE, RMSE
from sklearn.metrics import r2_score

logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

class EvaluateTFTModel:
    """
    Evaluate a TFT checkpoint on a long dataset using a sliding window to limit GPU memory usage.
    """

    def __init__(
        self,
        checkpoint_path: str,
        df: pd.DataFrame,
        params: dict,
        batch_size: int = 16,
        step_size: int = 48
    ):
        torch.set_float32_matmul_precision("medium")
        self.checkpoint_path = checkpoint_path
        self.df = df.copy()
        self.params = params
        self.batch_size = batch_size
        self.step_size = step_size
        self.encoder_len = params.get("max_encoder_length", 288)
        self.pred_len = params.get("max_prediction_length", 48)
        self.model = None
        self.reference_dataset = None

    def prepare_dataframe(self):
        self.df["Datetime"] = pd.to_datetime(self.df["Datetime"])
        self.df = self.df.sort_values("Datetime").reset_index(drop=True)
        self.df["time_idx"] = range(len(self.df))
        self.df["series_id"] = "France"

    def setup_reference_dataset(self):
        self.reference_dataset = rebuild_reference_dataset(self.df, self.params)

    def load_model(self):
        self.model = TemporalFusionTransformer.load_from_checkpoint(self.checkpoint_path)

    def sliding_evaluate(self):
        total_mae = total_rmse = total_r2 = total_coverage = n = 0

        for start in range(0, len(self.df) - self.encoder_len - self.pred_len, self.step_size):
            end = start + self.encoder_len + self.pred_len
            df_window = self.df.iloc[start:end].copy()

            dataset = TimeSeriesDataSet.from_dataset(
                self.reference_dataset,
                df_window,
                predict=True,
                stop_randomization=True
            )
            dataloader = dataset.to_dataloader(
                train=False,
                batch_size=self.batch_size,
                num_workers=15
            )

            with torch.no_grad():
                pred = self.model.predict(dataloader, mode="raw", return_x=True)

            y_true = pred.x["decoder_target"]
            y_low = pred.output.prediction[:, :, 0]
            y_high = pred.output.prediction[:, :, 2]

            total_mae += MAE()(y_high, y_true).item()
            total_rmse += RMSE()(y_high, y_true).item()
            total_r2 += r2_score(y_true.detach().cpu().numpy().flatten(), y_high.detach().cpu().numpy().flatten())
            total_coverage += ((y_true >= y_low) & (y_true <= y_high)).float().mean().item()
            n += 1

            del dataset, dataloader, pred
            torch.cuda.empty_cache()
            gc.collect()

        return {
            "MAE": total_mae / n,
            "RMSE": total_rmse / n,
            "R2": total_r2 / n,
            "Coverage": 100 * total_coverage / n
        }

    def run(self):
        self.prepare_dataframe()
        self.setup_reference_dataset()
        self.load_model()
        return self.sliding_evaluate()

In [3]:
import pandas as pd
from pathlib import Path
import logging

from eco2mix_add_features_minutes import Eco2mixFeaturesMinute

logger = logging.getLogger(__name__)

class Eco2mixXLSPreparer:
    """
    Prepares an Eco2mix .xls file for TFT evaluation.
    Steps:
    - Read raw file (tab-separated, latin1)
    - Keep only useful columns
    - Convert to datetime
    - Enrich with external features
    """
    def __init__(self, xls_path: str):
        self.xls_path = Path(xls_path)

    def read_df(self) -> pd.DataFrame:
        logger.info(f"Reading Eco2mix XLS file from {self.xls_path}")
        return pd.read_csv(self.xls_path, sep="\t", encoding="latin1", index_col=False, low_memory=False)

    def prepare(self) -> pd.DataFrame:
        df = self.read_df()

        # Filter and convert datetime
        df = df[["Date", "Heures", "Consommation"]].dropna()
        df["Datetime"] = pd.to_datetime(df["Date"].astype(str) + " " + df["Heures"], errors="coerce")
        df = df[["Datetime", "Consommation"]].dropna()

        # Optionnel : on ne garde que les minutes 00 et 30
        df = df[df["Datetime"].dt.minute.isin([0, 30])]
        df = df.sort_values("Datetime").reset_index(drop=True)

        logger.info(f"Loaded {len(df)} rows of 30-minute consumption data.")

        # External feature enrichment
        features = Eco2mixFeaturesMinute(df)
        df = features.run(include=["temperature", "sunshine", "weekday", "month", "season", "vacation"])
        logger.info("Added external features to Eco2mix data.")

        return df

In [4]:
import logging
import os

logging.getLogger("lightning.pytorch.utilities.rank_zero").setLevel(logging.WARNING)
logging.getLogger("lightning.fabric.utilities.device_parser").setLevel(logging.WARNING)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
os.environ["PL_DISABLE_PROG_BAR"] = "1"

# Prepare evaluation data from local XLS file
preparer = Eco2mixXLSPreparer("eCO2mix_RTE_Annuel-Definitif_2021.xls")
df = preparer.prepare()

# Parameters used to rebuild the dataset (should match training config)
params = {
    "max_encoder_length": 6 * 48,
    "max_prediction_length": 48,
    "known_reals": ["time_idx", "temperature", "is_vacation"],
    "unknown_reals": ["Consommation", "sunshine_duration"],
    "known_categoricals": ["weekday", "month", "season"],
}

# Load model from checkpoint and run evaluation on 2022 data
evaluator = EvaluateTFTModel(
    checkpoint_path="../data/06_models/tft/epoch=10-val_loss=790.21.ckpt",
    df=df,
    params=params,
    batch_size=1
)

# Run evaluation and print results (RMSE, MAE, R2, coverage)
results = evaluator.run()
print(results)

INFO:__main__:Reading Eco2mix XLS file from eCO2mix_RTE_Annuel-Definitif_2021.xls
INFO:__main__:Loaded 17520 rows of 30-minute consumption data.
INFO:root:Temperature features added.
INFO:root:Sunshine features added.
INFO:root:Weekday column added.
INFO:root:Month column added.
INFO:root:Season column added.
INFO:root:Vacation flag added.
INFO:__main__:Added external features to Eco2mix data.
/home/hassani/EPSI/MSPR/tft-env/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/hassani/EPSI/MSPR/tft-env/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
LOCAL_RANK: 0 - CUDA_

{'MAE': 4060.9274095929536, 'RMSE': 4474.317423388945, 'R2': -0.3306135949808792, 'Coverage': 79.1783070446684}
